# Pipeline Status Check

Scans `$BIDS_DIR` for `sub-*` directories and checks, for each subject, whether
the expected output file(s) for each pipeline stage exist. Writes a status CSV
to `derivatives/<OUTPUT_NAME>/<OUTPUT_NAME>.csv` and displays a colour-coded
summary table.

This notebook replaces the old `check_pipeline_outputs.py` +
`eg_check_stage.yml` combination — every setting (paths, sessions, tasks,
stages, patterns) is defined in the cells below instead of a separate YAML
file.

### How it works
1. **Settings** (Steps 1-4) define where to look and what to check for.
2. **Run the check** (Step 5) globs each stage's pattern for every
   subject x session x task combination.
3. **Results** (Step 6) merges with any existing CSV, saves it, and displays
   a colour-coded table + text summary.

Re-run the whole notebook any time to refresh the status. Subjects not listed
in `SUBJECTS` (Step 2) keep their previously recorded status.

---
## 1. Project paths

Sets `PROJECT` and sources `config/project_<name>.sh` (via `set_project.sh`)
to populate `BIDS_DIR`, `SUBJECTS_DIR`, etc. — same pattern as the other
walkthrough notebooks.

In [ ]:
import subprocess, os
from pathlib import Path

# -- Fill this in --------------------------------------------------------------
PROJECT = 'hypot'   # matches your config/project_<name>.sh file
# -------------------------------------------------------------------------------

result = subprocess.run(
    ['bash', '-c', f'source ~/.bash_profile && source "$PIPELINE_DIR/config/set_project.sh" {PROJECT} && env'],
    capture_output=True, text=True
)
for line in result.stdout.splitlines():
    if '=' in line:
        k, v = line.split('=', 1)
        os.environ[k] = v

BIDS_DIR     = str(Path(os.environ['BIDS_DIR']))
SUBJECTS_DIR = os.environ.get('SUBJECTS_DIR') or os.path.join(BIDS_DIR, 'derivatives', 'freesurfer')
CTX_DIR      = os.environ.get('CTX_DIR') or os.environ.get('PYCTX_DIR', '')

print(f"Project:     {os.environ.get('PROJ_NAME')}")
print(f"BIDS dir:    {BIDS_DIR}")
print(f"FreeSurfer:  {SUBJECTS_DIR}")
print(f"CTX dir:     {CTX_DIR or '(not set)'}")

---
## 2. Sessions, tasks & subjects to check

- `SESSIONS` / `TASKS` — used to expand stage columns whenever a stage's
  pattern contains `{ses}` / `{task}` (see Step 4).
- `SUBJECTS` — leave as `None` to check every `sub-*` directory found in
  `BIDS_DIR`, or give an explicit list (e.g. `['sub-01', 'sub-02']`) to limit
  the check to specific subjects (e.g. the one you just ran).

In [ ]:
SESSIONS = [
    'ses-01',
]

TASKS = [
    'pRFLE',
    'CSFLE',
]

SUBJECTS = None   # e.g. ['sub-01', 'sub-02'] to limit the check; None = all subjects

---
## 3. Output location

Status is written to `derivatives/<OUTPUT_NAME>/<OUTPUT_NAME>.csv`. If this
file already exists it is loaded, updated in place for the checked subjects,
and saved back — so other subjects' status is preserved.

In [ ]:
OUTPUT_NAME = 'pipeline_status'   # derivatives subdirectory + CSV file name

output_dir = os.path.join(BIDS_DIR, 'derivatives', OUTPUT_NAME)
os.makedirs(output_dir, exist_ok=True)
out_csv = os.path.join(output_dir, f'{OUTPUT_NAME}.csv')

print(f'Output CSV: {out_csv}')

---
## 4. Stages to check

Each stage defines a `pattern` — a glob, formatted with the placeholders
below, that should match **at least one file** if the stage is complete.

| Placeholder      | Resolves to |
|------------------|-------------|
| `{BIDS_DIR}`     | `BIDS_DIR` (Step 1) |
| `{SUBJECTS_DIR}` | `SUBJECTS_DIR` (Step 1) |
| `{CTX_DIR}`      | `CTX_DIR` (Step 1) — pycortex filestore |
| `{sub}`          | subject label, e.g. `sub-01` |
| `{ses}`          | session label, e.g. `ses-01` |
| `{task}`         | task label, e.g. `pRFLE` |

A stage is expanded into one column **per session** and/or **per task** only
when its pattern contains `{ses}` / `{task}`. `type` controls grouping in the
results table (`anatomical` -> `functional` -> `postproc`).

In [ ]:
STAGES = {

    # ---------- ANATOMICAL ------------------------------------------------
    'anat_01fmriprep': {
        'type': 'anatomical',
        'desc': 'fMRIPrep anat-only: T1w in MNI space',
        'pattern': '{BIDS_DIR}/derivatives/fmriprep/{sub}/anat/{sub}_*T1w.nii.gz',
    },
    'anat_02b14atlas': {
        'type': 'anatomical',
        'desc': 'Benson14 atlas labels projected onto FreeSurfer surface',
        'pattern': '{SUBJECTS_DIR}/{sub}/label/custom/lh.b14_*.label',
    },
    'anat_03pycortex': {
        'type': 'anatomical',
        'desc': 'Pycortex subject imported into filestore (requires CTX_DIR)',
        'pattern': '{CTX_DIR}/{sub}/',
    },

    # ---------- FUNCTIONAL -------------------------------------------------
    'func_01sdc': {
        'type': 'functional',
        'desc': 's01_sdc_AFNI: distortion-corrected BOLD timeseries',
        'pattern': '{BIDS_DIR}/derivatives/s1_AFNI_sdc/{sub}/{ses}/*task-{task}*sdc_bold.nii.gz',
    },
    'func_02coreg': {
        'type': 'functional',
        'desc': 's02_coreg: surface-projected (fsnative) BOLD timeseries',
        'pattern': '{BIDS_DIR}/derivatives/s2_coreg/{sub}/{ses}/*task-{task}*hemi-L*bold.func.gii',
    },
    'func_03fmriprep': {
        'type': 'functional',
        'desc': 's03_fmriprep_func: fMRIPrep confound timeseries',
        'pattern': '{BIDS_DIR}/derivatives/fmriprep/{sub}/{ses}/func/*task-{task}*confounds_timeseries.tsv',
    },
    'func_04confounds': {
        'type': 'functional',
        'desc': 's04_confounds: denoised surface BOLD timeseries '
                '(change desc-denoised -> desc-filtered for filter-only mode)',
        'pattern': '{BIDS_DIR}/derivatives/s4_denoised/{sub}/{ses}/*task-{task}*desc-denoised*bold.func.gii',
    },

    # ---------- POSTPROC ----------------------------------------------------
    'post_01prf': {
        'type': 'postproc',
        'desc': 's01_gauss_prfpy: Gaussian pRF iterative fit results',
        'pattern': '{BIDS_DIR}/derivatives/s5_gauss_prfpy/{sub}/{ses}/*task-pRF*model-gauss*stage-iter.csv',
    },
    'post_cf': {
        'type': 'postproc',
        'desc': 's03_cf_prfpy: connective field fit results',
        'pattern': '{BIDS_DIR}/derivatives/s6_cf_prfpy/{sub}/{ses}/*task-pRF*model-cf.csv',
    },
    'post_collate': {
        'type': 'postproc',
        'desc': 's04_collate_analyses: combined per-subject CSV',
        'pattern': '{BIDS_DIR}/derivatives/s7_compile/{sub}_combined.csv',
    },
}

---
## 5. Run the check

For each subject, every stage's pattern is formatted with the placeholders
above and globbed. A stage is marked **done (1)** if at least one file
matches, otherwise **0**.

In [ ]:
import glob
import pandas as pd


def find_subjects(bids_dir):
    """Sorted list of sub-* directories in bids_dir."""
    return sorted(
        os.path.basename(p)
        for p in glob.glob(os.path.join(bids_dir, 'sub-*'))
        if os.path.isdir(p)
    )


def check_pattern(pattern, sub, ses, task):
    """1 if the formatted glob finds >=1 match, else 0."""
    resolved = pattern.format(
        BIDS_DIR=BIDS_DIR, SUBJECTS_DIR=SUBJECTS_DIR, CTX_DIR=CTX_DIR,
        sub=sub, ses=ses or '', task=task or '',
    )
    return 1 if glob.glob(resolved) else 0


def stage_columns(stage_name, pattern):
    """(column_label, ses, task) for one stage, expanded over SESSIONS/TASKS
    only when the pattern uses {ses} / {task}."""
    needs_ses  = '{ses}'  in pattern
    needs_task = '{task}' in pattern

    if not needs_ses and not needs_task:
        return [(stage_name, None, None)]

    cols = []
    for ses in (SESSIONS if needs_ses else [None]):
        for task in (TASKS if needs_task else [None]):
            parts = [stage_name] + [p for p in (ses, f'task-{task}' if task else None) if p]
            cols.append(('_'.join(parts), ses, task))
    return cols


# anatomical -> functional -> postproc, preserving STAGES order within each type
TYPE_ORDER = {'anatomical': 0, 'functional': 1, 'postproc': 2}
stage_names = sorted(STAGES, key=lambda s: TYPE_ORDER.get(STAGES[s]['type'], 99))

col_spec = [
    (col, sname, ses, task)
    for sname in stage_names
    for col, ses, task in stage_columns(sname, STAGES[sname]['pattern'])
]
col_labels = [c[0] for c in col_spec]

all_subjects = find_subjects(BIDS_DIR)
subjects = SUBJECTS or all_subjects
subjects = [s if s.startswith('sub-') else f'sub-{s}' for s in subjects]

missing = [s for s in subjects if s not in all_subjects]
if missing:
    print(f'WARNING: not found in {BIDS_DIR}: {missing}')
    subjects = [s for s in subjects if s in all_subjects]

print(f'Subjects : {len(subjects)} ({", ".join(subjects)})')
print(f'Columns  : {len(col_labels)}')


rows = {}
for sub in subjects:
    row = {
        col: check_pattern(STAGES[sname]['pattern'], sub, ses, task)
        for col, sname, ses, task in col_spec
    }
    rows[sub] = row
    print(f'{sub}: {sum(row.values())}/{len(col_spec)} stages complete')

sub_check = pd.DataFrame.from_dict(rows, orient='index')
sub_check.index.name = 'subject'
sub_check = sub_check.sort_index()

In [ ]:
def highlight(val):
    if pd.isna(val):
        return 'background-color: #eeeeee; color: #999999'
    return ('background-color: #c6efce; color: #006100' if val == 1
            else 'background-color: #ffc7ce; color: #9c0006')

(
    sub_check[col_labels]
    .style
    .map(highlight)
    .format(lambda v: '' if pd.isna(v) else ('✓' if v == 1 else '✗'))
    .set_caption(f'Pipeline status — {out_csv}')
)

In [ ]:
summary = sub_check[col_labels].astype(float)

print(f'{"Stage":<55} {"Done":>6}  / N')
print('-' * 70)
for col in col_labels:
    n_done  = int(summary[col].sum())
    n_total = int(summary[col].notna().sum())
    bar = '#' * n_done + '.' * (n_total - n_done)
    print(f'{col:<55} {n_done:>6}  / {n_total:<4}  [{bar}]')